In [6]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Add path for data loader
sys.path.append(os.path.abspath('..'))
from src.data_loader import SP500DataLoader

print("✅ All imports ready")

✅ All imports ready


In [7]:
print("="*60)
print("LOADING DATA AND CREATING VOLATILITY LABELS")
print("="*60)

loader = SP500DataLoader(start_year=1997, end_year=2024)
df = loader.load_all()

# Extract log returns
all_returns = df['log_returns'].dropna()

# Use pre-COVID period for training (2010-2019)
train_returns = all_returns['2010':'2019'].values

print(f"Training data shape: {train_returns.shape}")

def create_volatility_sequences(data, lookback=60, vol_window=20):
    """
    Predict whether next day's volatility will be HIGH or LOW.
    
    Volatility: absolute return (|r|) as proxy
    Label: 1 if next day's volatility > recent average volatility
    """
    X, y = [], []
    
    for i in range(lookback, len(data) - 1):
        # Input: last 'lookback' returns
        X.append(data[i-lookback:i])
        
        # Calculate recent average volatility (last 'vol_window' days)
        recent_vol = np.abs(data[i-vol_window:i]).mean()
        
        # Next day volatility
        next_vol = np.abs(data[i+1])
        
        # Label: 1 if next volatility > recent average, else 0
        y.append(1 if next_vol > recent_vol else 0)
    
    return np.array(X), np.array(y)

lookback = 60
vol_window = 20

X, y = create_volatility_sequences(train_returns, lookback, vol_window)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance: HIGH volatility={y.sum()}, LOW volatility={len(y)-y.sum()}")
print(f"Proportion HIGH: {y.sum()/len(y)*100:.1f}%")

# Split
#X_train, X_val, y_train, y_val = train_test_split(
 #   X, y, test_size=0.2, random_state=42, shuffle=False
#)

# Scale features
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train.reshape(-1, lookback)).reshape(-1, lookback, 1)
#X_val_scaled = scaler.transform(X_val.reshape(-1, lookback)).reshape(-1, lookback, 1)

#print(f"Train shape: {X_train_scaled.shape}")
#print(f"Validation shape: {X_val_scaled.shape}")



LOADING DATA AND CREATING VOLATILITY LABELS
Training data shape: (2516,)
X shape: (2455, 60)
y shape: (2455,)
Class balance: HIGH volatility=992, LOW volatility=1463
Proportion HIGH: 40.4%


In [8]:
# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, lookback)).reshape(-1, lookback, 1)
X_val_scaled = scaler.transform(X_val.reshape(-1, lookback)).reshape(-1, lookback, 1)

print(f"Train shape: {X_train_scaled.shape}")
print(f"Validation shape: {X_val_scaled.shape}")

Train shape: (1964, 60, 1)
Validation shape: (491, 60, 1)


In [9]:
# ============================================================
# ADD CLASS WEIGHTS HERE
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {cls: weight for cls, weight in zip(classes, class_weights)}

print(f"\nClass weights: LOW (0) = {class_weight_dict[0]:.3f}, HIGH (1) = {class_weight_dict[1]:.3f}")
# Your output should show HIGH weight > LOW weight (e.g., 1.48 vs 0.84)


Class weights: LOW (0) = 0.834, HIGH (1) = 1.248


In [10]:
# Build model (same as before)
def build_volatility_lstm(lookback=60, lstm_units=32):
    model = Sequential([
        LSTM(lstm_units, input_shape=(lookback, 1)),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_volatility_lstm(lookback=lookback, lstm_units=32)

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train WITH class weights
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,  # ← ADD THIS LINE
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")

Epoch 1/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.5586 - loss: 0.6839 - val_accuracy: 0.6293 - val_loss: 0.6634
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.6059 - loss: 0.6673 - val_accuracy: 0.6497 - val_loss: 0.6438
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5998 - loss: 0.6631 - val_accuracy: 0.6415 - val_loss: 0.6404
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.6059 - loss: 0.6630 - val_accuracy: 0.6477 - val_loss: 0.6413
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.6034 - loss: 0.6608 - val_accuracy: 0.6517 - val_loss: 0.6427
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.5973 - loss: 0.6607 - val_accuracy: 0.6517 - val_loss: 0.6394
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.6034 - loss: 0.6599 - val_accuracy: 0.6497 - val_loss: 0.6391
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.6171 - loss: 0.6599 - val_accuracy: 0.6232 - v

In [11]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

# Predict on validation
y_pred_proba = model.predict(X_val_scaled, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n" + "="*60)
print("MODEL EVALUATION WITH CLASS WEIGHTS")
print("="*60)
print(f"Standard accuracy: {accuracy_score(y_val, y_pred):.2%}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_val, y_pred):.2%}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['LOW', 'HIGH']))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
print(f"\nConfusion Matrix:")
print(f"                Predicted LOW   Predicted HIGH")
print(f"Actual LOW:     {cm[0,0]:5d}           {cm[0,1]:5d}")
print(f"Actual HIGH:    {cm[1,0]:5d}           {cm[1,1]:5d}")


MODEL EVALUATION WITH CLASS WEIGHTS
Standard accuracy: 66.19%
Balanced accuracy: 64.76%

Classification Report:
              precision    recall  f1-score   support

         LOW       0.70      0.73      0.72       286
        HIGH       0.60      0.56      0.58       205

    accuracy                           0.66       491
   macro avg       0.65      0.65      0.65       491
weighted avg       0.66      0.66      0.66       491


Confusion Matrix:
                Predicted LOW   Predicted HIGH
Actual LOW:       210              76
Actual HIGH:       90             115


In [13]:
# ============================================================
# LOAD MONTE CARLO PATHS (CORRECTED FILE NAMES)
# ============================================================

import numpy as np
import os

# Path to simulated data
data_path = '../data/simulated/'

# Load all available Monte Carlo paths
normal_paths = np.load(data_path + 'mc_returns_normal.npy')
crisis_2008_paths = np.load(data_path + 'mc_returns_crisis_2008.npy')
crisis_covid_paths = np.load(data_path + 'mc_returns_crisis_covid.npy')
synthetic_paths = np.load(data_path + 'mc_returns_synthetic_extreme.npy')
squeeze_paths = np.load(data_path + 'mc_returns_squeeze.npy')
structural_break_paths = np.load(data_path + 'mc_returns_structural_break.npy')

print("✅ Monte Carlo paths loaded:")
print(f"   Normal:              {normal_paths.shape}")
print(f"   2008 Crisis:         {crisis_2008_paths.shape}")
print(f"   COVID Crisis:        {crisis_covid_paths.shape}")
print(f"   Synthetic Extreme:   {synthetic_paths.shape}")
print(f"   Non-linear Squeeze:  {squeeze_paths.shape}")
print(f"   Structural Break:    {structural_break_paths.shape}")

✅ Monte Carlo paths loaded:
   Normal:              (10000, 252)
   2008 Crisis:         (10000, 252)
   COVID Crisis:        (10000, 252)
   Synthetic Extreme:   (10000, 252)
   Non-linear Squeeze:  (10000, 252)
   Structural Break:    (10000, 252)


In [16]:
from sklearn.preprocessing import StandardScaler

# Combine all paths for fitting the scaler
all_mc_paths = np.vstack([
    normal_paths, 
    crisis_2008_paths, 
    crisis_covid_paths, 
    synthetic_paths,
    squeeze_paths,
    structural_break_paths
])

lookback = 60
scaler_mc = StandardScaler()
scaler_mc.fit(all_mc_paths[:, -lookback:])

print("✅ scaler_mc created and fitted on all Monte Carlo paths")

✅ scaler_mc created and fitted on all Monte Carlo paths


In [17]:
# ============================================================
# DEFINE HELPER FUNCTIONS
# ============================================================

def get_volatility_labels_mc(paths, lookback=60, vol_window=20):
    """
    Get true volatility labels for Monte Carlo paths.
    """
    n_paths = paths.shape[0]
    labels = []
    
    for i in range(n_paths):
        path = paths[i]
        recent_vol = np.abs(path[-vol_window:-1]).mean()
        next_vol = np.abs(path[-1])
        labels.append(1 if next_vol > recent_vol else 0)
    
    return np.array(labels)

def predict_on_paths_volatility(model, scaler, paths, lookback=60, n_paths=10000, batch_size=500):
    """
    Predict volatility regime on Monte Carlo paths.
    """
    n_paths = min(paths.shape[0], n_paths)
    all_probs = []
    
    for i in range(0, n_paths, batch_size):
        batch = paths[i:i+batch_size]
        features = batch[:, -lookback:]
        features_scaled = scaler.transform(features)
        features_scaled = features_scaled.reshape(-1, lookback, 1)
        probs = model.predict(features_scaled, verbose=0)
        all_probs.extend(probs.flatten())
        
        print(f"   Processed: {min(i+batch_size, n_paths)}/{n_paths}")
    
    return np.array(all_probs)

def calculate_accuracy(predictions, true_labels):
    """Calculate directional accuracy."""
    pred_labels = (predictions > 0.5).astype(int)
    return (pred_labels == true_labels).mean() * 100

In [18]:
# ============================================================
# TEST ON ALL MONTE CARLO SCENARIOS
# ============================================================

print("\n" + "="*70)
print("TESTING VOLATILITY LSTM ON MONTE CARLO PATHS")
print("="*70)

results = {}

# 1. Normal Scenario
print("\n1. Normal scenario...")
true_labels_normal = get_volatility_labels_mc(normal_paths)
probs_normal = predict_on_paths_volatility(model, scaler_mc, normal_paths, n_paths=10000)
acc_normal = calculate_accuracy(probs_normal, true_labels_normal)
results['Normal'] = acc_normal
print(f"   ✅ Accuracy: {acc_normal:.2f}%")

# 2. 2008 Crisis Scenario
print("\n2. 2008 Crisis scenario...")
true_labels_2008 = get_volatility_labels_mc(crisis_2008_paths)
probs_2008 = predict_on_paths_volatility(model, scaler_mc, crisis_2008_paths, n_paths=10000)
acc_2008 = calculate_accuracy(probs_2008, true_labels_2008)
results['2008 Crisis'] = acc_2008
print(f"   ✅ Accuracy: {acc_2008:.2f}%")

# 3. COVID Crisis Scenario
print("\n3. COVID Crisis scenario...")
true_labels_covid = get_volatility_labels_mc(crisis_covid_paths)
probs_covid = predict_on_paths_volatility(model, scaler_mc, crisis_covid_paths, n_paths=10000)
acc_covid = calculate_accuracy(probs_covid, true_labels_covid)
results['COVID Crisis'] = acc_covid
print(f"   ✅ Accuracy: {acc_covid:.2f}%")

# 4. Synthetic Extreme Scenario
print("\n4. Synthetic Extreme scenario...")
true_labels_synthetic = get_volatility_labels_mc(synthetic_paths)
probs_synthetic = predict_on_paths_volatility(model, scaler_mc, synthetic_paths, n_paths=10000)
acc_synthetic = calculate_accuracy(probs_synthetic, true_labels_synthetic)
results['Synthetic Extreme'] = acc_synthetic
print(f"   ✅ Accuracy: {acc_synthetic:.2f}%")

# 5. Non-linear Squeeze Scenario
print("\n5. Non-linear Squeeze scenario...")
true_labels_squeeze = get_volatility_labels_mc(squeeze_paths)
probs_squeeze = predict_on_paths_volatility(model, scaler_mc, squeeze_paths, n_paths=10000)
acc_squeeze = calculate_accuracy(probs_squeeze, true_labels_squeeze)
results['Non-linear Squeeze'] = acc_squeeze
print(f"   ✅ Accuracy: {acc_squeeze:.2f}%")

# 6. Structural Break Scenario
print("\n6. Structural Break scenario...")
true_labels_break = get_volatility_labels_mc(structural_break_paths)
probs_break = predict_on_paths_volatility(model, scaler_mc, structural_break_paths, n_paths=10000)
acc_break = calculate_accuracy(probs_break, true_labels_break)
results['Structural Break'] = acc_break
print(f"   ✅ Accuracy: {acc_break:.2f}%")


TESTING VOLATILITY LSTM ON MONTE CARLO PATHS

1. Normal scenario...
   Processed: 500/10000
   Processed: 1000/10000
   Processed: 1500/10000
   Processed: 2000/10000
   Processed: 2500/10000
   Processed: 3000/10000
   Processed: 3500/10000
   Processed: 4000/10000
   Processed: 4500/10000
   Processed: 5000/10000
   Processed: 5500/10000
   Processed: 6000/10000
   Processed: 6500/10000
   Processed: 7000/10000
   Processed: 7500/10000
   Processed: 8000/10000
   Processed: 8500/10000
   Processed: 9000/10000
   Processed: 9500/10000
   Processed: 10000/10000
   ✅ Accuracy: 52.29%

2. 2008 Crisis scenario...
   Processed: 500/10000
   Processed: 1000/10000
   Processed: 1500/10000
   Processed: 2000/10000
   Processed: 2500/10000
   Processed: 3000/10000
   Processed: 3500/10000
   Processed: 4000/10000
   Processed: 4500/10000
   Processed: 5000/10000
   Processed: 5500/10000
   Processed: 6000/10000
   Processed: 6500/10000
   Processed: 7000/10000
   Processed: 7500/10000
   Proc

In [19]:
import pandas as pd

results_df = pd.DataFrame(list(results.items()), columns=['Scenario', 'Accuracy (%)'])

print("\n" + "="*70)
print("FINAL RESULTS TABLE")
print("="*70)
print(results_df.to_string(index=False))

print("\n" + "="*70)
print("DEGRADATION ANALYSIS")
print("="*70)

baseline = results['Normal']
print(f"Baseline (Normal): {baseline:.2f}%")
print("-" * 40)

for scenario, acc in results.items():
    if scenario != 'Normal':
        change = acc - baseline
        if change < 0:
            print(f"{scenario:20} {acc:.2f}% → Change: {change:+.2f} pp (DEGRADATION)")
        else:
            print(f"{scenario:20} {acc:.2f}% → Change: {change:+.2f} pp (IMPROVEMENT)")

# Save results
import os
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/volatility_lstm_all_scenarios.csv', index=False)
print("\n✅ Results saved to results/volatility_lstm_all_scenarios.csv")


FINAL RESULTS TABLE
          Scenario  Accuracy (%)
            Normal         52.29
       2008 Crisis         53.38
      COVID Crisis         53.24
 Synthetic Extreme         53.55
Non-linear Squeeze         48.64
  Structural Break         51.89

DEGRADATION ANALYSIS
Baseline (Normal): 52.29%
----------------------------------------
2008 Crisis          53.38% → Change: +1.09 pp (IMPROVEMENT)
COVID Crisis         53.24% → Change: +0.95 pp (IMPROVEMENT)
Synthetic Extreme    53.55% → Change: +1.26 pp (IMPROVEMENT)
Non-linear Squeeze   48.64% → Change: -3.65 pp (DEGRADATION)
Structural Break     51.89% → Change: -0.40 pp (DEGRADATION)

✅ Results saved to results/volatility_lstm_all_scenarios.csv
